# DS 4320 Project 1 - Pipeline

## Breaking the Filter Bubble: A Diversity Aware Movie Recommendation System

### 1) Loading the four table relational database into DuckDB

Setting up Imports, Logging, and Configuration:

In [ ]:
# Importing necessary libraries:
import logging # Importing logging
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import duckdb
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Setting up Logging: writes to both console and log file in 'logs/pipeline.log'
os.makedirs('logs', exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    handlers=[
        logging.FileHandler('logs/pipeline.log'),
        logging.StreamHandler(sys.stdout)
    ]
)
log = logging.getLogger(__name__)

#  Configuration:
DATA_DIR = 'data/'
GENRE_COLS = [
    'Action','Adventure','Animation','Childrens','Comedy','Crime',
    'Documentary','Drama','Fantasy','Film_Noir','Horror','Musical',
    'Mystery','Romance','Sci_Fi','Thriller','War','Western'
]
N_CYCLES   = 25   # recommendation sessions to simulate
TOP_N      = 10   # items per recommendation list
N_NEIGHBORS = 20  # CF neighbors per user
TARGET_USERS = [1, 50, 100, 200, 300]  # users to simulate

log.info("Setup complete.")


## Step 1: Loading Data into DuckDB

The four .csv files are saved into DuckDB's in memory instance. 

In [ ]:
# Using try and except for error handling:
try:
    con = duckdb.connect()  # in-memory DuckDB instance
    for table, fname in [('ratings','ratings.csv'), ('movies','movies.csv'),
                         ('users','users.csv'), ('genres','genres.csv')]:
        # Create the table for each CSV file:
        con.execute(f"CREATE TABLE {table} AS SELECT * FROM read_csv_auto('{DATA_DIR}{fname}')")
        log.info(f"Loaded table '{table}' from {fname}")
    log.info("All 4 tables loaded into DuckDB.") # Completion message
except Exception as e:
    log.error(f"DuckDB load error: {e}")
    raise


## Step 2: SQL Queries - EDA

In [ ]:
# 1st Query: Finding the top 10 most rated movies:
try:
    # Selecting the 4 columns and calculating the number of ratings
    # and average rating for each movie:
    top_movies = con.execute("""
        SELECT m.movieId,
               m.title,
               COUNT(r.rating)       AS num_ratings,
               ROUND(AVG(r.rating), 2) AS avg_rating
        FROM   ratings r
        JOIN   movies  m ON r.movieId = m.movieId
        GROUP  BY m.movieId, m.title
        ORDER  BY num_ratings DESC
        LIMIT  10
    """).df()
    log.info("Query 1 (top movies) complete.")
    print("Top 10 Most-Rated Movies:")
    display(top_movies)
except Exception as e:
    log.error(f"Query 1 failed: {e}")
    raise

In [ ]:
# 2nd Query: Lookinga at Genre Popularity

# Counts how many movies belong to each genre using the binary flag columns. 
# This query helps understand genre distribution in the dataset and help interpret
# ILD Results

try:
    genre_rows = []
    for g in GENRE_COLS:
        # Counting how many movies have the genre flag set to 1 for each genre column
        count = con.execute(f'SELECT COUNT(*) FROM movies WHERE "{g}" = 1').fetchone()[0]
        genre_rows.append({'genre': g, 'movie_count': count}) # appending the genre and its count to the list
    genre_counts = pd.DataFrame(genre_rows).sort_values('movie_count', ascending=False)
    log.info("Query 2 (genre counts) complete.")
    print("Movies per Genre:")
    display(genre_counts)
except Exception as e:
    log.error(f"Query 2 failed: {e}")
    raise


In [ ]:
# 3rd Query: User Activity Distribution

# Identifies how many ratings each user has submitted, helps understand how
# active users produce more stable CF similarity estimates

try:
    user_activity = con.execute("""
        SELECT userId,
               COUNT(*)            AS num_ratings,
               ROUND(AVG(rating), 2) AS avg_rating
        FROM   ratings
        GROUP  BY userId
        ORDER  BY num_ratings DESC
    """).df()
    log.info("Query 3 (user activity) complete.")
    print(f"User activity: min={user_activity.num_ratings.min()}, "
          f"max={user_activity.num_ratings.max()}, "
          f"mean={user_activity.num_ratings.mean():.1f}")
    display(user_activity.head(10))
except Exception as e:
    log.error(f"Query 3 failed: {e}")
    raise


## Step 3: Load Tables into Pandas for ML Pipeline

In [ ]:
try:
    ratings = con.execute("SELECT * FROM ratings").df()
    movies  = con.execute("SELECT * FROM movies").df()
    users   = con.execute("SELECT * FROM users").df()
    movies_indexed = movies.set_index('movieId')  # fast lookup by movieId
    log.info(f"Loaded: ratings={ratings.shape}, movies={movies.shape}, users={users.shape}")
except Exception as e:
    log.error(f"Pandas load failed: {e}")
    raise


## Step 4: Collaborative Filtering and Diversity Functions

In [ ]:
# First function: build_user_item_matrix
# This function creates a user-item matrix from the ratings DataFrame, where rows represent users and columns represent movies. 
# The values in the matrix are the ratings given by users to movies. If a user has not rated a movie, the corresponding cell is filled with 0. 
# This is a common approach in collaborative filtering to handle missing ratings when calculating similarities between users or items.
def build_user_item_matrix(ratings_df):
    """
    Build a user × movie rating matrix.
    Missing values (movies a user hasn't rated) are filled with 0.
    Rationale: cosine similarity on sparse 0-filled vectors is standard in CF.
    """
    return ratings_df.pivot_table(
        index='userId', columns='movieId', values='rating'
    ).fillna(0)


# Second function: get_cf_recommendations
# This function implements user-based collaborative filtering to generate movie recommendations for a target user.
def get_cf_recommendations(target_user, user_item_matrix, ratings_df,
                            n_neighbors=N_NEIGHBORS, top_n=TOP_N*3):
    """
    User-based collaborative filtering.
    Finds the top-K most similar users, then ranks unseen movies
    by those neighbors' average ratings.
    Returns a list of top_n candidate movieIds.
    """
    try:
        user_vec = user_item_matrix.loc[[target_user]]
        sims = cosine_similarity(user_vec, user_item_matrix)[0]
        sim_series = pd.Series(sims, index=user_item_matrix.index).drop(target_user)
        top_neighbors = sim_series.nlargest(n_neighbors).index
        seen = set(ratings_df[ratings_df.userId == target_user].movieId)
        neighbor_ratings = ratings_df[ratings_df.userId.isin(top_neighbors)]
        candidates = (
            neighbor_ratings[~neighbor_ratings.movieId.isin(seen)]
            .groupby('movieId')['rating'].mean()
            .sort_values(ascending=False)
        )
        return candidates.head(top_n).index.tolist()
    except Exception as e:
        log.error(f"CF failed for user {target_user}: {e}")
        return []

# Third function: compute_ild
# This function calculates the Intra-List Diversity (ILD) of a list of recommended movies based on their genre vectors.
def compute_ild(movie_ids, movies_df, genre_cols):
    """
    Intra-List Diversity (ILD): mean pairwise genre cosine dissimilarity.
    Higher ILD = more genre-diverse recommendation list.
    Range: [0, 1]. Returns 0.0 if fewer than 2 items.
    """
    valid = [m for m in movie_ids if m in movies_df.index]
    if len(valid) < 2:
        return 0.0
    vecs = movies_df.loc[valid, genre_cols].values.astype(float)
    sim_mat = cosine_similarity(vecs)
    n = len(vecs)
    return float(np.mean([1 - sim_mat[i][j]
                           for i in range(n) for j in range(i+1, n)]))

# Fourth function: mmr_rerank
# This function implements Maximal Marginal Relevance (MMR) re-ranking to balance relevance and diversity in the recommendation list.
def mmr_rerank(candidates, movies_df, genre_cols, top_n=TOP_N, lam=0.5):
    """
    Maximal Marginal Relevance (MMR) re-ranking.
    Balances relevance (rank position) and diversity (genre dissimilarity).
    lam=0.5 gives equal weight to both objectives.
    Returns reranked list of top_n movieIds.
    """
    selected, remaining = [], list(candidates)
    while len(selected) < top_n and remaining:
        if not selected:
            selected.append(remaining.pop(0))
            continue
        best, best_score = None, -np.inf
        for mid in remaining:
            rel  = lam * (1.0 / (remaining.index(mid) + 1))
            if mid in movies_df.index and all(s in movies_df.index for s in selected):
                v   = movies_df.loc[mid, genre_cols].values.reshape(1,-1).astype(float)
                div = (1-lam) * min(
                    1 - cosine_similarity(v, movies_df.loc[s, genre_cols]
                                         .values.reshape(1,-1).astype(float))[0][0]
                    for s in selected
                )
            else:
                div = 0.0
            if rel + div > best_score:
                best_score, best = rel + div, mid
        if best:
            selected.append(best)
            remaining.remove(best)
    return selected

log.info("Helper functions defined.")


## Step 5: Filter Bubble Simulation (25 Cycles x 5 Users)

**Analysis Rationale:** Simulating 25 recommendation cycles for 5 representative users. For each cycle, a recommendation list using standard CF will be generated with its ILD computed. Then, we re rank the same candidates using MMR and computing ILD again.

Comparing the ILD across cycles reveals how quick each approach is narrowing down content diversity.

In [ ]:
try:
    log.info("Building user-item matrix...")
    user_item = build_user_item_matrix(ratings)

    log.info(f"Simulating {N_CYCLES} cycles for {len(TARGET_USERS)} users...")
    results = []
    for user in TARGET_USERS:
        log.info(f"  User {user}...")
        for cycle in range(N_CYCLES):
            cf_recs   = get_cf_recommendations(user, user_item, ratings)
            cf_ild    = compute_ild(cf_recs[:TOP_N], movies_indexed, GENRE_COLS)
            div_recs  = mmr_rerank(cf_recs, movies_indexed, GENRE_COLS)
            div_ild   = compute_ild(div_recs, movies_indexed, GENRE_COLS)
            results.append({'userId':user, 'cycle':cycle+1,
                            'cf_ild':cf_ild, 'diverse_ild':div_ild})
    results_df = pd.DataFrame(results)
    log.info("Simulation complete.")
    display(results_df.groupby('cycle')[['cf_ild','diverse_ild']].mean().round(3).head(10))
except Exception as e:
    log.error(f"Simulation error: {e}")
    raise


## Step 6: ML Model - Predict Low Diversity Users

**Analysis Rationale:** Training a Random Forest Classifier to predict which users are at the highest risk of filter bubble narrowing based on their activity level, average rating, age, and gender. This demonstrates that certain user profiles may be more susceptible to content narrowing which could help guide personalized diversity interventions in the future.

In [ ]:
try:
    # Feature engineering: merge user activity with demographics
    model_df = user_activity.merge(users[['userId','age','gender']], on='userId', how='left')
    model_df['gender_bin'] = (model_df['gender'] == 'M').astype(int)

    # Label: users whose average CF ILD falls below the median are "low diversity"
    avg_ild = results_df.groupby('userId')['cf_ild'].mean().reset_index()
    avg_ild.columns = ['userId', 'avg_cf_ild']
    median_ild = avg_ild['avg_cf_ild'].median()
    avg_ild['low_diversity'] = (avg_ild['avg_cf_ild'] < median_ild).astype(int)
    log.info(f"Median CF ILD threshold: {median_ild:.3f}")

    model_df = model_df.merge(avg_ild[['userId','low_diversity']], on='userId').dropna()
    features = ['num_ratings', 'avg_rating', 'age', 'gender_bin']
    X = model_df[features]
    y = model_df['low_diversity']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Random Forest — chosen for its ability to handle mixed feature types
    # and produce interpretable feature importances
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    log.info(f"Random Forest accuracy: {acc:.3f}")
    print(f"Accuracy: {acc:.3f}")
    print(classification_report(y_test, y_pred,
                                 target_names=['High Diversity','Low Diversity']))
except Exception as e:
    log.error(f"ML model error: {e}")
    raise


## Step 7: Visualization

Two panels are created here, the left one is the primary result of the ILD over 25 cycles, which directly answers the main question of if MMR re ranking reduces filter bubble formations. The right panel shows Random Forest feature importance, revealing which user characteristics most strongly predict susceptibility to content narrowing. 

In [ ]:
try:
    avg_by_cycle = results_df.groupby('cycle')[['cf_ild','diverse_ild']].mean()
    cycles   = avg_by_cycle.index.tolist()
    cf_ilds  = avg_by_cycle['cf_ild'].tolist()
    div_ilds = avg_by_cycle['diverse_ild'].tolist()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(
        'Filter Bubble Formation: Standard CF vs. Diversity-Aware Re-Ranking\n'
        'MovieLens 100K — Averaged Across 5 Users',
        fontsize=13, fontweight='bold'
    )

    # Left panel: ILD over cycles
    ax = axes[0]
    ax.plot(cycles, cf_ilds,  color='tomato',    lw=2.2, marker='o', ms=4,
            label='Standard CF (no intervention)')
    ax.plot(cycles, div_ilds, color='steelblue', lw=2.2, marker='s', ms=4,
            label='Diversity Re-ranking (MMR, λ=0.5)')
    ax.fill_between(cycles, cf_ilds, div_ilds,
                    alpha=0.12, color='green', label='Diversity gained')
    # FORMATTING:
    ax.set_xlabel('Recommendation Cycle (Session)', fontsize=11)
    ax.set_ylabel('Intra-List Diversity (ILD)', fontsize=11)
    ax.set_title('ILD Across Recommendation Cycles', fontsize=11)
    ax.set_ylim(0, 1.05)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    #  Right panel: Feature importance 
    ax2 = axes[1]
    importances = pd.Series(clf.feature_importances_, index=features).sort_values()
    importances.plot(kind='barh', ax=ax2, color='steelblue', edgecolor='white')
    ax2.set_title('Random Forest Feature Importance\n'
                  'Predicting Low-Diversity Recommendation Risk', fontsize=11)
    ax2.set_xlabel('Importance Score', fontsize=11)
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)

    plt.tight_layout()
    os.makedirs('pipeline', exist_ok=True)
    plt.savefig('pipeline/filter_bubble_chart.png', dpi=150, bbox_inches='tight')
    log.info("Chart saved to pipeline/filter_bubble_chart.png")
    plt.show()
except Exception as e:
    log.error(f"Visualization error: {e}")
    raise


## Results Summary

In [ ]:
print("FILTER BUBBLE SIMULATION RESULTS")

print(f"Dataset:          MovieLens 100K")
print(f"Users simulated:  {len(TARGET_USERS)}")
print(f"Cycles per user:  {N_CYCLES}")
print(f"List size (top-N): {TOP_N}")
print()
print(f"Average ILD — Standard CF:         {np.mean(cf_ilds):.4f}")
print(f"Average ILD — MMR Re-ranking:      {np.mean(div_ilds):.4f}")
print(f"Average diversity gain:            {np.mean(div_ilds) - np.mean(cf_ilds):.4f}")
print()
print(f"ILD at cycle  1 (CF vs MMR):  {cf_ilds[0]:.3f} vs {div_ilds[0]:.3f}")
print(f"ILD at cycle 25 (CF vs MMR):  {cf_ilds[-1]:.3f} vs {div_ilds[-1]:.3f}")
print()
print(f"RF Classifier Accuracy:  {acc:.3f}")

print("\nConclusion: MMR re-ranking consistently produces more")
print("genre-diverse recommendation lists than standard CF,")
print("demonstrating that a lightweight post-processing layer")
print("can measurably reduce filter bubble formation.")
log.info("Pipeline finished.")
